In [1]:
# ============================================================
# AŞAMA 1: KÜTÜPHANELER VE KLASÖR YOLLARI
# ------------------------------------------------------------
# Bu notebook sentetik FHSS / OFDM / DSSS sinyalleri üretir.
# Sonra bunları spektrogram görüntüsüne çevirip YOLO formatında kaydeder.
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random

# Rastgeleliği sabitleyelim ki sonuçlar tekrar üretilebilir olsun
np.random.seed(42)
random.seed(42)

# YOLO dataset ana klasörü
DATASET_DIR = Path("../data/spectrogram_yolo")

# Görüntü ve label klasörleri
IMG_TRAIN_DIR = DATASET_DIR / "images" / "train"
IMG_VAL_DIR   = DATASET_DIR / "images" / "val"

LBL_TRAIN_DIR = DATASET_DIR / "labels" / "train"
LBL_VAL_DIR   = DATASET_DIR / "labels" / "val"

# Klasörleri oluştur
for folder in [IMG_TRAIN_DIR, IMG_VAL_DIR, LBL_TRAIN_DIR, LBL_VAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Sınıflarımız
CLASSES = ["FHSS", "OFDM", "DSSS"]

print("Klasörler hazır.")
print("Dataset klasörü:", DATASET_DIR)
print("Sınıflar:", CLASSES)

Klasörler hazır.
Dataset klasörü: ..\data\spectrogram_yolo
Sınıflar: ['FHSS', 'OFDM', 'DSSS']


In [3]:
# ============================================================
# AŞAMA 2: GÜRÜLTÜ EKLEME FONKSİYONU
# ------------------------------------------------------------
# Bu fonksiyon temiz kompleks I/Q sinyale istenen SNR seviyesinde
# AWGN yani beyaz Gauss gürültüsü ekler.
#
# snr_db:
# 0 dB  -> gürültülü
# 10 dB -> orta
# 20 dB -> daha temiz
# ============================================================

def add_awgn(iq_signal, snr_db):
    # Sinyal gücü
    signal_power = np.mean(np.abs(iq_signal) ** 2)

    # SNR dB değerini lineer değere çevir
    snr_linear = 10 ** (snr_db / 10)

    # Gürültü gücü
    noise_power = signal_power / snr_linear

    # Kompleks gürültü üret
    noise = np.sqrt(noise_power / 2) * (
        np.random.randn(len(iq_signal)) + 1j * np.random.randn(len(iq_signal))
    )

    return iq_signal + noise

In [4]:
# ============================================================
# AŞAMA 3: SENTETİK FHSS ÜRETİCİ
# ------------------------------------------------------------
# FHSS: Frequency Hopping Spread Spectrum
#
# Mantık:
# Sinyal belirli süre bir frekansta kalır,
# sonra başka frekansa atlar.
#
# Spektrogramda basamak / merdiven benzeri izler oluşur.
# ============================================================

def generate_fhss_signal(
    total_len=4096,
    samples_per_hop=128,
    freqs=None,
    snr_db=15
):
    if freqs is None:
        # Normalize frekanslar: -0.5 ile +0.5 arası düşün
        freqs = [-0.35, -0.25, -0.15, -0.05, 0.08, 0.18, 0.30, 0.40]

    signal_parts = []

    num_hops = total_len // samples_per_hop

    for _ in range(num_hops):
        # Her hop için rastgele frekans seç
        f = np.random.choice(freqs)

        t = np.arange(samples_per_hop)

        # Kompleks taşıyıcı üret
        part = np.exp(1j * 2 * np.pi * f * t)

        # Hafif genlik değişimi ekleyelim
        amp = np.random.uniform(0.7, 1.2)
        part = amp * part

        signal_parts.append(part)

    iq = np.concatenate(signal_parts)

    # Tam uzunluğu garantiye al
    iq = iq[:total_len]

    # Gürültü ekle
    iq_noisy = add_awgn(iq, snr_db)

    return iq_noisy

In [ ]:
# ============================================================
# AŞAMA 4: SENTETİK OFDM ÜRETİCİ
# ------------------------------------------------------------
# OFDM: Orthogonal Frequency Division Multiplexing
#
# Mantık:
# Veri çok sayıda alt taşıyıcıya dağıtılır.
# IFFT ile zaman alanına çevrilir.
#
# Spektrogramda geniş bant blok gibi görünmesi beklenir.
# ============================================================

def qpsk_symbols(n):
    # QPSK sembolleri: 4 farklı faz
    bits_i = np.random.choice([-1, 1], size=n)
    bits_q = np.random.choice([-1, 1], size=n)
    return (bits_i + 1j * bits_q) / np.sqrt(2)


def generate_ofdm_signal(
    total_len=4096,
    n_subcarriers=64,
    active_ratio=0.75,
    cyclic_prefix=16,
    snr_db=15
):
    symbols = []

    symbol_len = n_subcarriers + cyclic_prefix

    num_symbols = total_len // symbol_len + 1

    active_count = int(n_subcarriers * active_ratio)

    for _ in range(num_symbols):
        # Frekans alanı taşıyıcıları
        freq_bins = np.zeros(n_subcarriers, dtype=complex)

        # Ortadaki alt taşıyıcıların bir kısmını aktif yap
        start = (n_subcarriers - active_count) // 2
        end = start + active_count

        freq_bins[start:end] = qpsk_symbols(active_count)

        # IFFT ile zaman alanına geç
        time_symbol = np.fft.ifft(np.fft.ifftshift(freq_bins))

        # Cyclic prefix ekle
        cp = time_symbol[-cyclic_prefix:]
        ofdm_symbol = np.concatenate([cp, time_symbol])

        symbols.append(ofdm_symbol)

    iq = np.concatenate(symbols)
    iq = iq[:total_len]

    # Güç normalize
    iq = iq / (np.sqrt(np.mean(np.abs(iq) ** 2)) + 1e-8)

    # Gürültü ekle
    iq_noisy = add_awgn(iq, snr_db)

    return iq_noisy

In [5]:
# ============================================================
# AŞAMA 5: SENTETİK DSSS ÜRETİCİ
# ------------------------------------------------------------
# DSSS: Direct Sequence Spread Spectrum
#
# Mantık:
# Veri biti, PN kodu ile çarpılarak daha geniş banda yayılır.
#
# Spektrogramda tek dar taşıyıcıdan daha yayvan enerji görülebilir.
# ============================================================

def generate_dsss_signal(
    total_len=4096,
    spreading_factor=16,
    snr_db=15
):
    # Kaç veri biti gerekir?
    num_bits = total_len // spreading_factor + 1

    # BPSK veri bitleri
    bits = np.random.choice([-1, 1], size=num_bits)

    # PN kodu
    pn_code = np.random.choice([-1, 1], size=(num_bits, spreading_factor))

    # Her biti spreading_factor kadar yay
    spread_signal = bits[:, None] * pn_code

    # Düzleştir
    baseband = spread_signal.reshape(-1)

    # Uzunluğu ayarla
    baseband = baseband[:total_len]

    # Kompleks hale getir
    iq = baseband.astype(np.float32) + 0j

    # Hafif frekans kayması ekleyelim
    t = np.arange(len(iq))
    freq_offset = np.random.uniform(-0.08, 0.08)
    iq = iq * np.exp(1j * 2 * np.pi * freq_offset * t)

    # Güç normalize
    iq = iq / (np.sqrt(np.mean(np.abs(iq) ** 2)) + 1e-8)

    # Gürültü ekle
    iq_noisy = add_awgn(iq, snr_db)

    return iq_noisy

In [6]:
# ============================================================
# AŞAMA 6: SPEKTROGRAM MATRİSİ OLUŞTURMA
# ------------------------------------------------------------
# Kompleks I/Q sinyali alır.
# Kısa zamanlı FFT uygular.
# Zaman-frekans matrisi döndürür.
# ============================================================

def create_spectrogram_matrix(
    iq_signal,
    fft_size=256,
    hop_size=64
):
    spec_list = []

    for start in range(0, len(iq_signal) - fft_size + 1, hop_size):
        window = iq_signal[start:start + fft_size]

        # Pencereleme
        windowed = window * np.hanning(fft_size)

        # FFT
        spectrum = np.fft.fftshift(np.fft.fft(windowed))

        # dB güç
        power_db = 20 * np.log10(np.abs(spectrum) + 1e-8)

        spec_list.append(power_db)

    spec = np.array(spec_list)

    # Çıktı shape: zaman x frekans
    return spec

In [7]:
# ============================================================
# AŞAMA 7: SPEKTROGRAMI GÖRÜNTÜ OLARAK KAYDETME
# ------------------------------------------------------------
# YOLO eğitimi için eksensiz, temiz .png görüntü üretir.
#
# Not:
# YOLO için genelde eksen, başlık, colorbar istemeyiz.
# Sadece sinyal görüntüsü olmalı.
# ============================================================

def save_spectrogram_image(spec, save_path):
    plt.figure(figsize=(4, 4))

    # Spektrogramı görüntü olarak çiziyoruz
    plt.imshow(
        spec.T,
        aspect="auto",
        origin="lower",
        cmap="viridis"
    )

    # YOLO için eksenleri kapatıyoruz
    plt.axis("off")

    # Kenar boşluklarını kaldırıyoruz
    plt.tight_layout(pad=0)

    # Kaydet
    plt.savefig(save_path, dpi=120, bbox_inches="tight", pad_inches=0)

    plt.close()

In [8]:
# ============================================================
# AŞAMA 8: YOLO LABEL DOSYASI OLUŞTURMA
# ------------------------------------------------------------
# YOLO label formatı:
# class_id x_center y_center width height
#
# Tüm değerler 0-1 arasında normalize edilir.
#
# İlk prototipte görüntüde tek sinyal olduğunu varsayıyoruz.
# Bu yüzden kutuyu tüm görüntüyü kapsayacak şekilde veriyoruz.
# ============================================================

def save_yolo_label(label_path, class_id):
    # Tüm görüntüyü kapsayan bbox
    x_center = 0.5
    y_center = 0.5
    width = 1.0
    height = 1.0

    with open(label_path, "w") as f:
        f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

In [9]:
# ============================================================
# AŞAMA 9: YOLO SPEKTROGRAM VERİ SETİ ÜRETME
# ------------------------------------------------------------
# Bu hücre FHSS / OFDM / DSSS için spektrogram görüntüleri üretir.
#
# İlk prototip:
# Her sınıf için 120 train, 30 val görüntü
#
# Sonra artırabiliriz.
# ============================================================

def generate_dataset_for_class(
    class_name,
    class_id,
    generator_func,
    n_train=120,
    n_val=30,
    total_len=4096
):
    print(f"\nÜretiliyor: {class_name}")

    # Train üret
    for i in range(n_train):
        # Farklı SNR seviyeleri kullanıyoruz
        snr_db = np.random.choice([0, 5, 10, 15, 20])

        # I/Q sinyal üret
        iq = generator_func(total_len=total_len, snr_db=snr_db)

        # Spektrogram oluştur
        spec = create_spectrogram_matrix(iq, fft_size=256, hop_size=64)

        # Dosya adları
        img_name = f"{class_name.lower()}_train_{i:04d}.png"
        lbl_name = f"{class_name.lower()}_train_{i:04d}.txt"

        img_path = IMG_TRAIN_DIR / img_name
        lbl_path = LBL_TRAIN_DIR / lbl_name

        # Görüntü ve label kaydet
        save_spectrogram_image(spec, img_path)
        save_yolo_label(lbl_path, class_id)

    # Validation üret
    for i in range(n_val):
        snr_db = np.random.choice([0, 5, 10, 15, 20])

        iq = generator_func(total_len=total_len, snr_db=snr_db)
        spec = create_spectrogram_matrix(iq, fft_size=256, hop_size=64)

        img_name = f"{class_name.lower()}_val_{i:04d}.png"
        lbl_name = f"{class_name.lower()}_val_{i:04d}.txt"

        img_path = IMG_VAL_DIR / img_name
        lbl_path = LBL_VAL_DIR / lbl_name

        save_spectrogram_image(spec, img_path)
        save_yolo_label(lbl_path, class_id)

    print(f"{class_name} tamamlandı.")


# Sınıf üretici eşleşmeleri
generators = {
    "FHSS": generate_fhss_signal,
    "OFDM": generate_ofdm_signal,
    "DSSS": generate_dsss_signal
}

# Her sınıf için üret
for class_id, class_name in enumerate(CLASSES):
    generate_dataset_for_class(
        class_name=class_name,
        class_id=class_id,
        generator_func=generators[class_name],
        n_train=120,
        n_val=30,
        total_len=4096
    )

print("\nTüm YOLO spektrogram veri seti üretildi.")

NameError: name 'generate_ofdm_signal' is not defined

In [10]:
# ============================================================
# EKSİK FONKSİYON: SENTETİK OFDM ÜRETİCİ
# ------------------------------------------------------------
# Hata sebebi:
# generate_ofdm_signal fonksiyonu tanımlanmadan Aşama 9 çalıştırılmış.
#
# Bu hücreyi çalıştırınca OFDM üretici fonksiyonu belleğe yüklenir.
# ============================================================

def qpsk_symbols(n):
    # QPSK için basit kompleks semboller üretir
    bits_i = np.random.choice([-1, 1], size=n)
    bits_q = np.random.choice([-1, 1], size=n)

    # Gücü normalize etmek için sqrt(2)'ye bölüyoruz
    return (bits_i + 1j * bits_q) / np.sqrt(2)


def generate_ofdm_signal(
    total_len=4096,
    n_subcarriers=64,
    active_ratio=0.75,
    cyclic_prefix=16,
    snr_db=15
):
    # OFDM sembollerini burada biriktireceğiz
    symbols = []

    # Bir OFDM sembolünün toplam uzunluğu
    symbol_len = n_subcarriers + cyclic_prefix

    # İstenen toplam uzunluğu dolduracak kadar sembol üret
    num_symbols = total_len // symbol_len + 1

    # Aktif alt taşıyıcı sayısı
    active_count = int(n_subcarriers * active_ratio)

    for _ in range(num_symbols):
        # Frekans alanında boş alt taşıyıcılar
        freq_bins = np.zeros(n_subcarriers, dtype=complex)

        # Ortadaki taşıyıcıları aktif yapıyoruz
        start = (n_subcarriers - active_count) // 2
        end = start + active_count

        # Aktif taşıyıcılara QPSK sembolleri koyuyoruz
        freq_bins[start:end] = qpsk_symbols(active_count)

        # IFFT ile zaman alanına geçiyoruz
        time_symbol = np.fft.ifft(np.fft.ifftshift(freq_bins))

        # Cyclic Prefix ekliyoruz
        cp = time_symbol[-cyclic_prefix:]
        ofdm_symbol = np.concatenate([cp, time_symbol])

        symbols.append(ofdm_symbol)

    # Tüm OFDM sembollerini birleştir
    iq = np.concatenate(symbols)

    # İstenen uzunluğa kırp
    iq = iq[:total_len]

    # Güç normalize
    iq = iq / (np.sqrt(np.mean(np.abs(iq) ** 2)) + 1e-8)

    # AWGN gürültüsü ekle
    iq_noisy = add_awgn(iq, snr_db)

    return iq_noisy


print("generate_ofdm_signal fonksiyonu tanımlandı.")

generate_ofdm_signal fonksiyonu tanımlandı.


In [11]:
# ============================================================
# FONKSİYONLAR TANIMLI MI KONTROLÜ
# ------------------------------------------------------------
# Bu hücre üç üretici fonksiyonun bellekte olup olmadığını kontrol eder.
# ============================================================

print("FHSS fonksiyonu:", generate_fhss_signal)
print("OFDM fonksiyonu:", generate_ofdm_signal)
print("DSSS fonksiyonu:", generate_dsss_signal)

FHSS fonksiyonu: <function generate_fhss_signal at 0x000001A3EBEEA480>
OFDM fonksiyonu: <function generate_ofdm_signal at 0x000001A3EC0E2F20>
DSSS fonksiyonu: <function generate_dsss_signal at 0x000001A3EBEEA520>
